In [20]:
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.


In [21]:
import pandas as pd
import xgboost as xgb
import os

df = pd.read_csv("data/creditcard.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [22]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 227845
Testing samples: 56962


In [23]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=(
        y_train.value_counts()[0] / y_train.value_counts()[1]
    ),
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Create the models folder inside the project
os.makedirs("models", exist_ok=True)

# Save the trained model
model.save_model("models/xgboost_fraud_model.json")

print("Model saved successfully.")

[0]	validation_0-logloss:0.60965
[50]	validation_0-logloss:0.03192
[100]	validation_0-logloss:0.00911
[150]	validation_0-logloss:0.00439
[199]	validation_0-logloss:0.00317
Model saved successfully.


In [24]:
# Select 800 legitimate and 200 fraudulent transactions
sample_normal = df[df["Class"] == 0].sample(
    n=800,
    random_state=42
)

sample_fraud = df[df["Class"] == 1].sample(
    n=200,
    random_state=42
)

# Combine and shuffle the transactions
sample_df = pd.concat(
    [sample_normal, sample_fraud]
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Ensure the data folder exists
os.makedirs("data", exist_ok=True)

# Save the dashboard sample
sample_df.to_csv(
    "data/dashboard_sample.csv",
    index=False
)

print("Dashboard sample created successfully!")
print("Dataset shape:", sample_df.shape)
print("Class distribution:")
print(sample_df["Class"].value_counts())

Dashboard sample created successfully!
Dataset shape: (1000, 31)
Class distribution:
Class
0    800
1    200
Name: count, dtype: int64


In [25]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Generate predictions on the held-out test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(
    "Average Precision Score:",
    round(average_precision_score(y_test, y_prob), 4)
)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.83      0.84        98

    accuracy                           1.00     56962
   macro avg       0.92      0.91      0.92     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
[[56849    15]
 [   17    81]]
Average Precision Score: 0.8655


In [26]:
# Model evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(
    "Average Precision Score:",
    round(average_precision_score(y_test, y_prob), 4)
)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.83      0.84        98

    accuracy                           1.00     56962
   macro avg       0.92      0.91      0.92     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
[[56849    15]
 [   17    81]]
Average Precision Score: 0.8655
